In [1]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster  import DBSCAN
from cymetric.pointgen.pointgen import PointGenerator

In [25]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from scipy.optimize import root
from cymetric.pointgen.pointgen import PointGenerator
from scipy.optimize import least_squares

# reproducibility
np.random.seed(7)

# ----------------------------------------------------------------------
# parameters
# ----------------------------------------------------------------------
NUM_SEEDS   = 5000     # initial quintic samples to seed solver
TOL_ROOT    = 1e-6     # convergence tolerance for root-finding
MAXFEV      = 200      # max function evaluations for root-finder
NN_K        = 20       # neighbors for local PCA
DBS_MINPTS  = 5        # DBSCAN min_samples

# ----------------------------------------------------------------------
# 1) Sample raw quintic points (z1..z4 in patch Z0=1)
# ----------------------------------------------------------------------
mon = 5 * np.eye(5, dtype=int)
pg  = PointGenerator(mon, np.ones(5), np.ones(1), np.array([4]))

def sample_quintic(n=NUM_SEEDS):
    Z = pg.generate_points(n)        # (n,5) complex on Σ Z_i^5=0
    return Z[:,1:]                   # drop Z0=1 coordinate, return (n,4)

# ----------------------------------------------------------------------
# 2) SLAG residuals: Lagrangian & calibration
# ----------------------------------------------------------------------
def fs_metric(z):
    # Fubini–Study metric on patch z0=1, coords z1..z4
    sq = np.vdot(z, z).real
    return ((1+sq)*np.eye(4) - np.outer(np.conj(z), z)) / (1+sq)**2


def slag_residual(z):
    """
    z: complex array length-4 (z1..z4)
    returns (R_lag, R_calib)
    """
    # ---- FS metric ----
    G = fs_metric(z)

    # ---- tangent directions: kernel of dF (1 eqn) => complex 3-dim kernel, real 6-dim
    gradF = 5 * z**4                  # shape (4,)
    A     = np.vstack([gradF.real, gradF.imag])  # (2,4)
    _,_,Vh = np.linalg.svd(A)
    T2    = Vh[-2:].T                # (4,2) complex basis
    Mreal = np.hstack([T2.real, T2.imag])  # (4,4) real
    Ur,_,_ = np.linalg.svd(Mreal)
    V3_real   = Ur[:,:3]             # (4,3) real basis
    # convert each real basis vector to complex by adding 0j
    V3 = [V3_real[:,i].astype(complex) for i in range(3)]

    # ---- Lagrangian residual: max |ω(v_a,v_b)| ----
    R_lag = 0.0
    for i in range(3):
        for j in range(i+1, 3):
            val = np.imag(np.vdot(V3[i], G.dot(V3[j])))
            R_lag = max(R_lag, abs(val))

    # ---- calibration residual: |Im Ω(v1,v2,v3)| ----
    mu = - (1 + np.sum(z**5))
    z4 = mu**(1/5)
    M3 = np.vstack([V3[k][:3].real for k in range(3)]).T  # (3,3)
    omega_eval = np.linalg.det(M3) / (5 * z4**4)
    R_cal = abs(omega_eval.imag)

    return R_lag, R_cal

# ----------------------------------------------------------------------
# 3) Solver: Newton-root find for SLAG eqns
# ----------------------------------------------------------------------

def solve_slag(z0):
    def fvec(x):
        z = x[:4] + 1j*x[4:]
        return np.array(slag_residual(z))        # length-2

    x0  = np.hstack([z0.real, z0.imag])
    sol = least_squares(fvec, x0, xtol=TOL_ROOT, ftol=TOL_ROOT,
                        max_nfev=MAXFEV)
    if sol.success and np.max(np.abs(sol.fun)) < 1e-6:
        return sol.x[:4] + 1j*sol.x[4:]
    return None


# ----------------------------------------------------------------------
# 4) Generate seeds & solve
# ----------------------------------------------------------------------
z_seeds = sample_quintic()
solutions = []
for z0 in z_seeds:
    z_sol = solve_slag(z0)
    if z_sol is not None:
        solutions.append(z_sol)
print(f"Converged SLAG points: {len(solutions)}")

# convert to real-imag packed format for clustering
X_slag = []
for z in solutions:
    full = np.hstack(([1+0j], z))
    X_slag.append(np.column_stack([full.real, full.imag]).ravel())
X_slag = np.array(X_slag)

# ----------------------------------------------------------------------
# 5) Local PCA rank-3 test
# ----------------------------------------------------------------------
nbrs_s = NearestNeighbors(n_neighbors=NN_K+1).fit(X_slag)
_, nn_s = nbrs_s.kneighbors(X_slag)

eig_ratio = np.empty(len(X_slag))
for i in range(len(X_slag)):
    C   = np.cov(X_slag[nn_s[i,1:]].T)
    lam = np.sort(np.linalg.eigvalsh(C))[::-1]
    eig_ratio[i] = lam[3] / lam[0]
cand_idx = np.where(eig_ratio < 1e-2)[0]
print("rank-3 candidates:", len(cand_idx))

# ----------------------------------------------------------------------
# 6) Projector features & DBSCAN clustering
# ----------------------------------------------------------------------
proj_feats = []
for i in cand_idx:
    C, vecs = np.cov(X_slag[nn_s[i,1:]].T), None
    _, vecs = np.linalg.eigh(C)
    V3 = vecs[:, -3:]
    proj_feats.append((V3 @ V3.T).ravel())

feat = np.hstack([
    X_slag[cand_idx],      # coords
    np.vstack(proj_feats)  # projector features
])
feat = StandardScaler().fit_transform(feat)

nbr_feat = NearestNeighbors(n_neighbors=DBS_MINPTS+1).fit(feat)
dists,_ = nbr_feat.kneighbors(feat)
eps0 = np.percentile(dists[:,DBS_MINPTS], 90)
eps  = eps0
clusters = []
while eps <= 5*eps0 and not clusters:
    labels   = DBSCAN(eps=eps, min_samples=DBS_MINPTS).fit_predict(feat)
    clusters = [cand_idx[labels==l] for l in set(labels) if l!=-1]
    eps    *= 1.4
if not clusters:
    raise RuntimeError("No clusters found – try more seeds or different tolerances")
clusters_sizes = [len(c) for c in clusters]
print("Candidate cluster sizes:", clusters_sizes)


Converged SLAG points: 0


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.